# Jigsaw Toxic Comment Classification — Baselines (Multi-Label)

This notebook trains three classic baselines on the Jigsaw dataset:
- **Naive Bayes**
- **Logistic Regression**
- **Linear SVM**

### Why this version is different from a "toxic vs. non-toxic" setup

The competition is **multi-label**, not binary and not multi-class. Every comment gets a
**0 or 1 for each of 6 labels at the same time**:

`toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`

A comment can be `toxic=1` **and** `insult=1` **and** `obscene=1` all at once. So instead of
training *one* classifier that picks *one* class, we train **6 independent binary classifiers**
— one per label, each answering "yes/no" for that label only. Scikit-learn does this for us
automatically with `OneVsRestClassifier`.

### What you need
- `train.csv`, `test.csv`, `test_labels.csv` from the [Kaggle competition page](https://www.kaggle.com/competitions/jigsaw-toxic-comment-classification-challenge/data)
- Either upload them directly, or use the Kaggle API cell below.


## 1. Setup

In [ ]:
# If you're on Colab and don't have the data yet, you have two options:
#
# OPTION A — Upload manually:
#   Run the cell below and select train.csv, test.csv, test_labels.csv from your computer.
#
# OPTION B — Kaggle API (faster, no manual upload):
#   1. Go to kaggle.com -> Account -> Create New API Token (downloads kaggle.json)
#   2. Run the "Kaggle API" cell instead and upload kaggle.json when prompted.

# ---- OPTION A: manual upload ----
from google.colab import files
uploaded = files.upload()  # select train.csv, test.csv, test_labels.csv (can multi-select)


In [ ]:
# ---- OPTION B: Kaggle API (skip this cell if you used Option A above) ----
# from google.colab import files
# files.upload()  # upload your kaggle.json here
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle competitions download -c jigsaw-toxic-comment-classification-challenge
# !unzip -o jigsaw-toxic-comment-classification-challenge.zip -d data
# !unzip -o "data/train.csv.zip" -d data
# !unzip -o "data/test.csv.zip" -d data
# !unzip -o "data/test_labels.csv.zip" -d data


In [ ]:
import numpy as np
import pandas as pd
import re
import string
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load data

`train.csv` has the comments **and** the 6 label columns.
`test.csv` has comments only. `test_labels.csv` has the true labels for `test.csv`,
released after the competition ended (some rows are `-1`, meaning "not used for
scoring" — we drop those when evaluating on test).


In [ ]:
# Adjust these paths if your files ended up somewhere else (e.g. "data/train.csv")
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
TEST_LABELS_PATH = "test_labels.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
test_labels_df = pd.read_csv(TEST_LABELS_PATH)

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
train_df.head()


In [ ]:
# Quick look at how imbalanced each label is.
# This matters a lot for a multi-label problem: most comments are NOT toxic at all,
# and within toxic comments, some labels (like "threat") are rare.
label_counts = train_df[LABEL_COLS].sum().sort_values(ascending=False)
label_counts.plot(kind="bar", figsize=(8, 4), title="Positive examples per label")
plt.ylabel("count")
plt.show()

print(label_counts)
print(f"\n% of comments with NO label at all (clean): "
      f"{(train_df[LABEL_COLS].sum(axis=1) == 0).mean() * 100:.1f}%")


## 3. Text cleaning

Nothing fancy — lowercase, strip newlines/IP-addresses/extra punctuation. Keep it light:
TF-IDF and linear models work fine on lightly-cleaned text, and over-cleaning can throw
away signal (e.g. repeated punctuation like "!!!" or all-caps can correlate with toxicity).


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", " ", text)  # strip IP addresses
    text = re.sub(r"http\S+|www\S+", " ", text)  # strip URLs
    text = re.sub(r"[^a-z0-9\s']", " ", text)  # strip punctuation except apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["comment_text"].astype(str).apply(clean_text)
test_df["clean_text"] = test_df["comment_text"].astype(str).apply(clean_text)

train_df[["comment_text", "clean_text"]].head(3)


## 4. Train / validation split

We split `train.csv` ourselves to get a validation set, since the real `test.csv` labels
are only used for final reporting (and some rows are masked with -1).


In [ ]:
from sklearn.model_selection import train_test_split

X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_df["clean_text"],
    train_df[LABEL_COLS].values,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("train:", X_train_text.shape[0], " val:", X_val_text.shape[0])


## 5. TF-IDF features

We turn text into numeric vectors with TF-IDF (term frequency–inverse document frequency).
Word-level + character-level n-grams together usually work best for toxic comment text,
because character n-grams catch misspellings people use to dodge filters (e.g. "id*ot").


In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=40000,
    min_df=2,
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=40000,
    min_df=2,
)

from scipy.sparse import hstack

print("Fitting word vectorizer...")
X_train_word = word_vectorizer.fit_transform(X_train_text)
X_val_word = word_vectorizer.transform(X_val_text)

print("Fitting char vectorizer...")
X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)

X_train = hstack([X_train_word, X_train_char]).tocsr()
X_val = hstack([X_val_word, X_val_char]).tocsr()

print("Feature matrix shape (train):", X_train.shape)


## 6. Models

Each model below is wrapped in `OneVsRestClassifier`. This trains **6 separate binary
classifiers internally** (one per label) and lets each predict independently — exactly
what multi-label classification needs. No extra setup required on our end beyond passing
the full `(n_samples, 6)` label matrix.

A note on `LinearSVC`: it doesn't output probabilities by default (just a decision score),
so we wrap it in `CalibratedClassifierCV` to get probability outputs — needed for the
ROC-AUC metric the competition actually uses.


In [ ]:
def evaluate_model(name, model, X_val, y_val, label_cols):
    """Prints per-label ROC-AUC and the mean column-wise AUC (the competition metric)."""
    y_pred_proba = model.predict_proba(X_val)

    # OneVsRestClassifier with predict_proba returns shape (n_samples, n_labels) directly
    aucs = []
    print(f"--- {name} ---")
    for i, col in enumerate(label_cols):
        auc = roc_auc_score(y_val[:, i], y_pred_proba[:, i])
        aucs.append(auc)
        print(f"  {col:15s} AUC: {auc:.4f}")
    mean_auc = np.mean(aucs)
    print(f"  {'MEAN AUC':15s}: {mean_auc:.4f}\n")
    return mean_auc, y_pred_proba


In [ ]:
# ---- Naive Bayes ----
nb_model = OneVsRestClassifier(MultinomialNB())
nb_model.fit(X_train, y_train)
nb_auc, nb_proba = evaluate_model("Naive Bayes", nb_model, X_val, y_val, LABEL_COLS)


In [ ]:
# ---- Logistic Regression ----
logreg_model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, C=4.0, random_state=RANDOM_STATE)
)
logreg_model.fit(X_train, y_train)
logreg_auc, logreg_proba = evaluate_model("Logistic Regression", logreg_model, X_val, y_val, LABEL_COLS)


In [ ]:
# ---- Linear SVM ----
# LinearSVC has no predict_proba, so we calibrate it to get probability-like scores.
svm_base = LinearSVC(C=1.0, random_state=RANDOM_STATE, max_iter=5000)
svm_model = OneVsRestClassifier(
    CalibratedClassifierCV(svm_base, method="sigmoid", cv=3)
)
svm_model.fit(X_train, y_train)
svm_auc, svm_proba = evaluate_model("Linear SVM", svm_model, X_val, y_val, LABEL_COLS)


## 7. Compare all three baselines

In [ ]:
results = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "Linear SVM"],
    "Mean AUC": [nb_auc, logreg_auc, svm_auc],
}).sort_values("Mean AUC", ascending=False).reset_index(drop=True)

results


In [ ]:
results.plot(x="Model", y="Mean AUC", kind="bar", legend=False, figsize=(6, 4),
             title="Baseline comparison (mean column-wise ROC-AUC)")
plt.ylim(0.8, 1.0)
plt.ylabel("Mean AUC")
plt.xticks(rotation=0)
plt.show()


## 8. Per-label comparison (bonus, optional)

Useful for your report: shows which labels are hardest for which model. Typically `threat`
and `identity_hate` are the lowest-scoring labels for every model, because they have the
fewest positive training examples.


In [ ]:
per_label_results = pd.DataFrame({
    "label": LABEL_COLS,
    "Naive Bayes": [roc_auc_score(y_val[:, i], nb_proba[:, i]) for i in range(6)],
    "Logistic Regression": [roc_auc_score(y_val[:, i], logreg_proba[:, i]) for i in range(6)],
    "Linear SVM": [roc_auc_score(y_val[:, i], svm_proba[:, i]) for i in range(6)],
})

per_label_results.set_index("label").plot(kind="bar", figsize=(10, 5),
                                            title="Per-label AUC by model")
plt.ylabel("AUC")
plt.ylim(0.7, 1.0)
plt.xticks(rotation=30)
plt.show()

per_label_results


## 9. Predict on the real test set (optional — for Kaggle submission format)

This produces a `submission.csv` in the exact format Kaggle expects: one probability
per label, per test comment.


In [ ]:
# Pick whichever model performed best above. Logistic Regression is a common winner here.
best_model = logreg_model  # change to nb_model / svm_model if a different one wins for you

X_test_word = word_vectorizer.transform(test_df["clean_text"])
X_test_char = char_vectorizer.transform(test_df["clean_text"])
X_test = hstack([X_test_word, X_test_char]).tocsr()

test_proba = best_model.predict_proba(X_test)

submission = pd.DataFrame(test_proba, columns=LABEL_COLS)
submission.insert(0, "id", test_df["id"])
submission.to_csv("submission_baseline.csv", index=False)

submission.head()


## Summary

- This notebook treats the problem correctly as **multi-label**: 6 independent yes/no
  predictions per comment, not one class out of 6.
- `OneVsRestClassifier` is what makes a normally-binary model (NB, LogReg, SVM) work
  in this multi-label setting — it trains one binary copy of the model per label.
- The competition metric is the **mean of the per-label ROC-AUC scores**, which is what
  `evaluate_model()` reports above.
- Next step: compare these numbers against your teammate's BiLSTM and your DistilBERT
  results using the same mean-AUC metric, so all models are judged the same way.
